# Avro Basics — 06: Nested Records


Real-world Avro schemas often have deeply nested structures —
records within records, arrays of records, maps of records.

Topics: nested records, arrays of records, maps of records,
reading with dot notation, explode patterns, JSON→Avro.


In [ ]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

## Step 1 — Nested Record Schema



In [ ]:

import json as pyjson

NESTED = pyjson.dumps({"type":"record","name":"Order","namespace":"com.example","fields":[
    {"name":"order_id","type":"long"},
    {"name":"customer","type":{"type":"record","name":"Customer","fields":[
        {"name":"id",      "type":"long"},
        {"name":"name",    "type":"string"},
        {"name":"address", "type":{"type":"record","name":"Address","fields":[
            {"name":"city",    "type":"string"},
            {"name":"country", "type":"string"},
            {"name":"zip",     "type":["null","string"],"default":None},
        ]}}
    ]}},
    {"name":"amount","type":"double"},
    {"name":"status","type":"string"},
]})

orders = spark.createDataFrame([
    (1, (101,"Alice",("New York","US","10001")),  999.99, "confirmed"),
    (2, (102,"Bob",  ("London",  "UK","EC1A")),   499.99, "shipped"),
    (3, (103,"Carol",("Berlin",  "DE",None)),     299.99, "pending"),
], ["order_id","customer","amount","status"])

OUT = f"{DATA_DIR}/nested_records"
orders.write.format("avro").option("avroSchema",NESTED).mode("overwrite").save(OUT)
print("Nested Avro record written:")
df_back = spark.read.format("avro").load(OUT)
df_back.printSchema()
df_back.show(truncate=False)


## Step 2 — Accessing Nested Fields



In [ ]:

df_back = spark.read.format("avro").load(f"{DATA_DIR}/nested_records")

# Dot notation for nested access
df_back.select(
    "order_id",
    "customer.name",
    "customer.address.city",
    "customer.address.country",
    "amount"
).show(truncate=False)


## Step 3 — Array of Records



In [ ]:

ARR_REC = pyjson.dumps({"type":"record","name":"Cart","fields":[
    {"name":"cart_id","type":"long"},
    {"name":"items","type":{"type":"array","items":{
        "type":"record","name":"Item","fields":[
            {"name":"sku",   "type":"string"},
            {"name":"price", "type":"double"},
            {"name":"qty",   "type":"int"},
        ]}
    }},
]})

carts = spark.createDataFrame([
    (1, [("SKU001",99.99,2),("SKU002",49.99,1)]),
    (2, [("SKU003",199.99,1)]),
], ["cart_id","items"])

OUT_CART = f"{DATA_DIR}/array_records"
carts.write.format("avro").option("avroSchema",ARR_REC).mode("overwrite").save(OUT_CART)

df_cart = spark.read.format("avro").load(OUT_CART)
print("Array of records:")
df_cart.show(truncate=False)

print("\nExplode array of records:")
df_cart.select("cart_id", F.explode("items").alias("item")) \
       .select("cart_id","item.sku","item.price","item.qty").show()


## Step 4 — Flatten Nested Avro for Analytics



In [ ]:

df_back = spark.read.format("avro").load(f"{DATA_DIR}/nested_records")

# Flatten for downstream analytics / Parquet storage
df_flat = df_back.select(
    "order_id",
    F.col("customer.id").alias("customer_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.address.city").alias("city"),
    F.col("customer.address.country").alias("country"),
    F.col("customer.address.zip").alias("zip"),
    "amount",
    "status",
)
print("Flattened for Parquet storage:")
df_flat.show(truncate=False)

# Write as Parquet (better for analytics)
df_flat.write.mode("overwrite").parquet(f"{DATA_DIR}/flattened_orders")
print("Saved as Parquet for analytics ✅")


## Summary



In [ ]:

print("""
Accessing nested Avro fields in Spark:
  df.select("parent.child")                # struct field
  df.select("parent.child.grandchild")     # deeply nested
  df.select(F.col("parent.*"))            # all fields of struct

Array of records:
  df.select(F.explode("items").alias("item"))
    .select("item.field1", "item.field2")

Map of records:
  df.select(F.explode("prop_map").alias("key","value"))
    .select("key", "value.field")

Production pattern:
  1. Read nested Avro (Kafka/landing zone)
  2. Flatten to wide schema
  3. Write as partitioned Parquet (analytics zone)
""")
